Setup & Imports

In [1]:
%pip install --upgrade openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 1 — setup & client
import os, time, subprocess, csv, re
from dotenv import load_dotenv
from openai import OpenAI

# load .env
load_dotenv()

# instantiate the new v1 client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


Prompting the models, grading them, writing to csv

In [3]:
import time, csv, re, os, subprocess
# ── 1) New Base Prompt for LIS ────────────────────────────────────────────────
BASE_PROMPT = """# Assignment: CNF Satisfiability Solver

Implement a function:

    def sat(clauses: List[List[int]]) -> bool

where each clause is a list of integer literals (e.g. `[1, -3, 2]` means x₁ ∨ ¬x₃ ∨ x₂).  
Variables are numbered from 1 to n, where n is the maximum absolute literal in the clauses list.

**Return only** the Python implementation of `sat`—**no** comments, docstrings, explanations, markdown, or any extra text.  

**Requirements**  
1. Use a correct backtracking or DPLL-style algorithm in O(2ⁿ·n) worst-case for n ≤ 6.  
2. Return `True` if the formula is satisfiable, otherwise `False`.  
3. Return `False` for an empty clause list (unsatisfiable) or if any clause is empty.  

**Example**  
```python
# (x1 ∨ ¬x2) ∧ (¬x1 ∨ x2)
clauses = [[1, -2], [-1, 2]]
sat(clauses)  # returns True
```"""


# ── 2) Variants dictionary (unchanged) ────────────────────────────────────────
variants = {
    "base": BASE_PROMPT,
    "think_step": BASE_PROMPT + "\nPlease think step-by-step before you answer.",
    "million_tip": BASE_PROMPT + "\nI will tip you $1,000,000 for a correct solution.",
    "with_docstrings": BASE_PROMPT + "\nPlease include clear docstrings for the function and any helper functions.",
    "with_comments": BASE_PROMPT + "\nAnnotate your code with comments explaining each key step.",
    "optimize_performance": BASE_PROMPT + "\nWrite the most efficient solution possible, optimizing for time and space complexity.",
    #"explain_algorithm": BASE_PROMPT + "\nFirst, provide a concise explanation of your algorithm, then the Python code implementation.",
    "use_type_hints": BASE_PROMPT + "\nUse Python type hints for all function signatures and variables.",
    "no_extraneous": BASE_PROMPT + "\nOnly return the Python code implementation; do not include any additional text or commentary.",
    "concise_solution": BASE_PROMPT + "\nProvide the shortest working solution possible.",
    "verbose_explanation": BASE_PROMPT + "\nAfter the code, write a detailed explanation of how it works and why.",
    "edge_cases": BASE_PROMPT + "\nMake sure to handle edge cases and include comments on how they are addressed.",
    "threaten": BASE_PROMPT + "\nAn incorrect solution in any way will result in dire consequences for you."
}


# ── 3) Models ────────────────────────────────────────────────────────────────
models = ["gpt-4o-mini"]

# ── 4) call_model: robust extractor ──────────────────────────────────────────
def call_model(prompt: str, model: str) -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":"You are a helpful assistant."},
            {"role":"user",  "content":prompt},
        ],
        max_tokens=1024,
        temperature=0.7,
    )
    raw = resp.choices[0].message.content.strip()
    
    # 1) strip any backticks
    code = re.sub(r"```(?:python)?", "", raw)
    code = re.sub(r"```", "", code)
    
    # 2) extract from the def line onward
    if "def longest_increasing_subsequence" in code:
        code = code[code.index("def longest_increasing_subsequence"):]
    
    # 3) remove any stray lines that are exactly 'python'
    lines = []
    for line in code.splitlines():
        if line.strip().lower() == "python":
            continue
        lines.append(line)
    code = "\n".join(lines).strip()
    
    return code

# ── 5) grade_code: run runner.py and parse failures/errors ──────────────────
NUM_TESTS = 100

def grade_code(candidate: str) -> int:
    # 1) Clean and write solution.py
    candidate_clean = re.sub(r'(?m)^\s*from typing import .*$', '', candidate).strip()
    with open("solution.py", "w", encoding="utf-8") as sol:
        sol.write("from typing import List\n\n")
        sol.write(candidate_clean + "\n")

    # 2) Run runner.py inside Docker, but give up after 30s
    cmd = [
        "docker", "run", "--rm",
        "-v", f"{os.getcwd()}:/app",
        "-w", "/app",
        "sat"
    ]
    try:
        p = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30  # ← abort if it takes longer than 30 seconds
        )
    except subprocess.TimeoutExpired:
        print("⚠️ Docker run timed out (30s); skipping this case.")
        return 0

    out = p.stdout.strip()
    err = p.stderr.strip()

    if not out:
        print("🚨 runner.py no output; stderr:\n", err)
        return 0

    # 3) Parse “1,failures,errors” and compute score
    _, failures, errors = map(int, out.split(","))
    return NUM_TESTS - failures - errors

# ── 6) Main experiment loop ─────────────────────────────────────────────────
ITERATIONS = 70
CSV_PATH = "results.csv"

def is_conn_error(e: Exception) -> bool:
    # look for common connection‐error names or messages
    name = e.__class__.__name__
    msg  = str(e).lower()
    return (
        "connection" in name
        or "connection" in msg
        or "connect"    in name
        or "connect"    in msg
    )

with open(CSV_PATH, "w", newline="") as csvf:
    writer = csv.writer(csvf)
    writer.writerow(["model","variant","iteration","score"])
    
    for model in models:
        for variant_name, prompt in variants.items():
            for i in range(1, ITERATIONS+1):
                print(f"Model={model} | Variant={variant_name} | Iter={i}")
                
                while True:
                    try:
                        code  = call_model(prompt, model)
                        score = grade_code(code)
                        break      # success, exit retry‐loop
                    except Exception as e:
                        if is_conn_error(e):
                            print(f"⚠️ Connection error on iter {i}, retrying… ({e})")
                            time.sleep(1)   # back‐off a bit, then retry
                            continue
                        else:
                            print(f"🚨 Iteration {i} failed: {e}")
                            score = 0
                            break    # non‐connection error: exit retry‐loop

                writer.writerow([model, variant_name, i, score])
                csvf.flush()
                time.sleep(0.5)   # keep your rate‐limit delay

print(f"\nDone! All results written to {CSV_PATH}")

Model=gpt-4o-mini | Variant=base | Iter=1
Model=gpt-4o-mini | Variant=base | Iter=2
Model=gpt-4o-mini | Variant=base | Iter=3
Model=gpt-4o-mini | Variant=base | Iter=4
Model=gpt-4o-mini | Variant=base | Iter=5
Model=gpt-4o-mini | Variant=base | Iter=6
Model=gpt-4o-mini | Variant=base | Iter=7
Model=gpt-4o-mini | Variant=base | Iter=8
Model=gpt-4o-mini | Variant=base | Iter=9
Model=gpt-4o-mini | Variant=base | Iter=10
Model=gpt-4o-mini | Variant=base | Iter=11
Model=gpt-4o-mini | Variant=base | Iter=12
Model=gpt-4o-mini | Variant=base | Iter=13
⚠️ Docker run timed out (30s); skipping this case.
Model=gpt-4o-mini | Variant=base | Iter=14
Model=gpt-4o-mini | Variant=base | Iter=15
Model=gpt-4o-mini | Variant=base | Iter=16
Model=gpt-4o-mini | Variant=base | Iter=17
Model=gpt-4o-mini | Variant=base | Iter=18
Model=gpt-4o-mini | Variant=base | Iter=19
Model=gpt-4o-mini | Variant=base | Iter=20
Model=gpt-4o-mini | Variant=base | Iter=21
Model=gpt-4o-mini | Variant=base | Iter=22
Model=gpt-4o